In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PATH / "feature_dataset.csv")

print(df.shape)

(63000, 65)


In [2]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,M20,M21,M22,M40,M41,M42,M43,M60,M61,M62,...,SpectralFlatness,SpectralRollOff,SpectralBandwidth,MphiNL,SigmaDP,SigmaZ2,M2M4SNR,Modulation,OriginalSNR,Label
0,0.000026,0.000069,0.000026,3.157343e-09,2.930586e-09,6.460820e-09,2.930586e-09,3.500275e-13,4.633123e-13,4.327737e-13,...,0.627377,43,18.381460,1.0,3.797474,0.005159,1.105342,QPSK,2,6
1,0.000028,0.000068,0.000028,3.215990e-09,2.804059e-09,6.498267e-09,2.804059e-09,1.781424e-13,4.586850e-13,3.564625e-13,...,0.630078,48,19.898510,1.0,2.716261,0.005688,0.889030,QPSK,2,6
2,0.000004,0.000069,0.000004,2.910366e-09,8.069788e-10,7.002352e-09,8.069788e-10,1.588626e-13,3.876018e-13,1.401655e-13,...,0.624587,49,19.985046,1.0,3.927797,0.005326,0.360494,QPSK,2,6
3,0.000005,0.000070,0.000005,3.598735e-09,7.073600e-10,7.253493e-09,7.073600e-10,3.053956e-13,5.095078e-13,1.243595e-13,...,0.606380,47,19.097059,1.0,2.835475,0.006038,0.163557,QPSK,2,6
4,0.000018,0.000068,0.000018,2.224453e-09,2.107830e-09,6.237477e-09,2.107830e-09,1.804345e-13,2.696272e-13,2.822643e-13,...,0.602295,43,19.311969,1.0,2.253482,0.005549,1.231920,QPSK,2,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62995,0.000012,0.000069,0.000012,9.836777e-10,1.269483e-09,6.593998e-09,1.269483e-09,1.501898e-13,1.406296e-13,1.540585e-13,...,0.476958,26,14.640144,1.0,6.712201,0.006895,1.186821,QAM64,6,5
62996,0.000029,0.000078,0.000029,1.711707e-09,5.023596e-09,1.082102e-08,5.023596e-09,4.183721e-13,4.287749e-13,9.831322e-13,...,0.258891,10,11.059364,1.0,2.435119,0.008228,-2.663121,QAM64,6,5
62997,0.000038,0.000072,0.000038,1.782190e-09,5.098510e-09,8.094923e-09,5.098510e-09,1.997625e-13,3.189114e-13,7.638134e-13,...,0.390499,22,14.217020,1.0,5.094953,0.007605,-0.435454,QAM64,6,5
62998,0.000028,0.000074,0.000028,1.439306e-09,4.804105e-09,9.163291e-09,4.804105e-09,4.699322e-13,3.696777e-13,8.861841e-13,...,0.277886,12,11.051652,1.0,5.081371,0.007793,-1.422118,QAM64,6,5


In [3]:
# Fill NaNs only for M2M4SNR
if "M2M4SNR" in df.columns:
    median_value = df["M2M4SNR"].median()
    df["M2M4SNR"] = df["M2M4SNR"].fillna(median_value)

print(df.isnull().sum().sum())
print(df.shape)

0
(63000, 65)


In [4]:
duplicate_columns = [
    "M22",
    "C20",
    "C21",
    "C80",
    "C84",
    "GammaMean",
    "GammaMax"
]

duplicate_columns = [
    col for col in duplicate_columns
    if col in df.columns
]

df.drop(columns=duplicate_columns, inplace=True)

print(df.shape)

(63000, 58)


In [5]:
constant_columns = [
    "WaveletASKCorrelation"
]

constant_columns = [
    col for col in constant_columns
    if col in df.columns
]

df.drop(columns=constant_columns, inplace=True)

print(df.shape)

(63000, 57)


In [6]:
redundant = [
    "M40",
    "M41",
    "M42",
    "M43",
    "M60",
    "M61",
    "M62",
    "M63",
    "M80",
    "M84"
]

redundant = [
    col for col in redundant
    if col in df.columns
]

df.drop(columns=redundant, inplace=True)

print(df.shape)

(63000, 47)


In [7]:
print(df.isnull().sum().sum())

print(np.isinf(df.select_dtypes(np.number)).sum().sum())

print(df.shape)

0
0
(63000, 47)


In [8]:
df.to_csv(
    DATA_PATH / "feature_dataset_clean.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [9]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH / "feature_dataset_clean.csv")

X = df.drop(columns=["Modulation","Label"])

y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(50400, 45)
(12600, 45)


In [10]:
X_train.to_csv(DATA_PATH / "X_train.csv", index=False)
X_test.to_csv(DATA_PATH / "X_test.csv", index=False)

y_train.to_frame("Label").to_csv(DATA_PATH / "y_train.csv", index=False)
y_test.to_frame("Label").to_csv(DATA_PATH / "y_test.csv", index=False)